# Notebook 18 — Physics-Informed Cascaded DO Model

Professor suggestion: decouple DO into a temperature-driven baseline (Henry's Law)
plus a residual correction term. Two setups:
- Setup A: full EA-inspired version with Optuna-tuned residual LSTM
- Setup B: minimal version — global linear fit, AR(7) temperature, temp-only residual LSTM

Reference: OC_f = (a × T_f + b) + LSTM(residual)

In [ ]:
import sys, warnings, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
warnings.filterwarnings('ignore')

REPO_ROOT = Path('/storage/homefs/tn20y076/AareML')
sys.path.insert(0, str(REPO_ROOT))

from src.config import (
    FEATURES, TARGETS, LOOKBACK, HORIZON,
    TRAIN_END, VAL_END, SEED, FOCUS_GAUGE,
    RESULTS_DIR, FIGURES_DIR,
)
from src.data import load_gauge, preprocess, train_val_test_split, make_windows
from src.model import Seq2SeqLSTM, RiverDataset, train_model, predict, get_y_true

np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

DO_IDX = list(TARGETS).index('O2C_sensor')
TEMP_IDX = list(FEATURES).index('temp_sensor')

# Optuna settings for Setup A
N_TRIALS_A = 30

# Default hyperparameters for Setup B
HIDDEN_B, LAYERS_B, DROPOUT_B = 64, 1, 0.2
LR_B, BATCH_B = 1e-3, 256

## 1. Load Data

In [ ]:
raw = load_gauge(FOCUS_GAUGE)
data = preprocess(raw)
train_df, val_df, test_df = train_val_test_split(data)
train_means = (pd.concat([train_df[FEATURES].mean(), train_df[TARGETS].mean()])
               .groupby(level=0).first())

X_tr, y_tr, _ = make_windows(train_df, train_means, features=FEATURES, targets=TARGETS)
X_va, y_va, _ = make_windows(val_df,   train_means, features=FEATURES, targets=TARGETS)
X_te, y_te, dates_te = make_windows(test_df,  train_means, features=FEATURES, targets=TARGETS)

print(f"Train: {X_tr.shape}, Val: {X_va.shape}, Test: {X_te.shape}")
y_do_true = y_te[:, :, DO_IDX]   # (N, H) — ground truth DO
print(f"DO true range: [{y_do_true.min():.2f}, {y_do_true.max():.2f}] mg/L")

## 2. Temperature Forecasts (T_f)

In [ ]:
# ── T_f for Setup A: use nb04b LSTM temperature predictions ──────────────
# Load if available, otherwise use AR(7) as fallback
temp_results_path = RESULTS_DIR / 'temp_transfer_results.csv'

# Retrain temperature LSTM for focus gauge
from src.config import FEATURES_TEMP, TARGETS_TEMP
raw_temp = load_gauge(FOCUS_GAUGE)
data_temp = preprocess(raw_temp)
tr_t, va_t, te_t = train_val_test_split(data_temp)
tmeans = (pd.concat([tr_t[FEATURES_TEMP].mean(), tr_t[TARGETS_TEMP].mean()])
          .groupby(level=0).first())

X_tr_t, y_tr_t, _ = make_windows(tr_t, tmeans, features=FEATURES_TEMP, targets=TARGETS_TEMP)
X_va_t, y_va_t, _ = make_windows(va_t, tmeans, features=FEATURES_TEMP, targets=TARGETS_TEMP)
X_te_t, y_te_t, _ = make_windows(te_t, tmeans, features=FEATURES_TEMP, targets=TARGETS_TEMP)

N_t, L_t, F_t = X_tr_t.shape; H_t, T_t = y_tr_t.shape[1], y_tr_t.shape[2]
fsc_t = StandardScaler().fit(X_tr_t.reshape(-1, F_t))
tsc_t = StandardScaler().fit(y_tr_t.reshape(-1, T_t))

def sc_t(X, y, N): 
    return (fsc_t.transform(X.reshape(-1,F_t)).reshape(N,L_t,F_t).astype('float32'),
            tsc_t.transform(y.reshape(-1,T_t)).reshape(N,H_t,T_t).astype('float32'))

Xs_tr_t, ys_tr_t = sc_t(X_tr_t, y_tr_t, N_t)
Xs_va_t, ys_va_t = sc_t(X_va_t, y_va_t, X_va_t.shape[0])
Xs_te_t, ys_te_t = sc_t(X_te_t, y_te_t, X_te_t.shape[0])

dl_tr_t = torch.utils.data.DataLoader(RiverDataset(Xs_tr_t, ys_tr_t), batch_size=256, shuffle=True)
dl_va_t = torch.utils.data.DataLoader(RiverDataset(Xs_va_t, ys_va_t), batch_size=256, shuffle=False)
ds_te_t = RiverDataset(Xs_te_t, ys_te_t)

mdl_t = Seq2SeqLSTM(n_feat=F_t, n_tgt=T_t, hidden=128, n_layers=1, dropout=0.2).to(DEVICE)
mdl_t, _ = train_model(mdl_t, dl_tr_t, dl_va_t, device=DEVICE, epochs=30, lr=1e-3, patience=5, verbose=False)

T_f_A = predict(mdl_t, ds_te_t, tsc_t, device=DEVICE)[:, :, 0]  # (N, H) — Setup A temperature forecast
rmse_temp = float(np.sqrt(np.mean((T_f_A - y_te_t[:,:,0])**2))  )
print(f"Setup A T_f LSTM RMSE: {rmse_temp:.3f}°C")

# ── T_f for Setup B: AR(7) on temperature ─────────────────────────────────
temp_series = data['temp_sensor'].ffill().bfill().values
temp_train  = data.loc[:TRAIN_END, 'temp_sensor'].ffill().bfill().values
ar_fit = ARIMA(temp_train, order=(7, 0, 0)).fit()

n_test_start = len(train_df) + len(val_df)
N_te = X_te.shape[0]
T_f_B = np.zeros((N_te, HORIZON))
for i in range(N_te):
    abs_idx = n_test_start + i + LOOKBACK
    history = temp_series[max(0, abs_idx-200): abs_idx]
    try:
        T_f_B[i] = ar_fit.apply(history, refit=False).forecast(steps=HORIZON)
    except:
        T_f_B[i] = history[-1]

rmse_b = float(np.sqrt(np.mean((T_f_B - y_te[:, :, TEMP_IDX])**2)))
print(f"Setup B T_f AR(7) RMSE: {rmse_b:.3f}°C")

## 3. Linear DO Baseline: OC_f_0 = a × T_f + b

In [ ]:
# Training data: true DO vs true temperature windows
temp_train_windows = X_tr[:, -1, TEMP_IDX]  # last observed temperature per window
do_train_targets   = y_tr[:, :, DO_IDX]     # (N_tr, H) true DO

# ── Setup A: per-gauge (here just gauge 2473) ─────────────────────────────
# We'll use the LSTM T_f for test, but for the linear fit we use true T
# Fit: DO_horizon_h ~ T_future_h for each horizon h separately
A_coefs = np.zeros((HORIZON, 2))  # [a, b] per horizon
for h in range(HORIZON):
    X_lin = temp_train_windows.reshape(-1, 1)  # crude: use last T as proxy
    y_lin = do_train_targets[:, h]
    reg = LinearRegression().fit(X_lin, y_lin)
    A_coefs[h] = [reg.coef_[0], reg.intercept_]

# Apply to test: use T_f_A
last_obs_temp_te = X_te[:, -1, TEMP_IDX]
OC_f_0_A = np.column_stack([
    A_coefs[h, 0] * T_f_A[:, h] + A_coefs[h, 1]
    for h in range(HORIZON)
])  # (N_te, H)
rmse_lin_A = float(np.sqrt(np.mean((OC_f_0_A - y_do_true)**2)))
print(f"Setup A linear baseline RMSE: {rmse_lin_A:.4f} mg/L")

# ── Setup B: global fit across all hours/windows ─────────────────────────
# Flatten: each (window, horizon) pair is one sample
T_flat = temp_train_windows.repeat(HORIZON)    # crude global T proxy
DO_flat = do_train_targets.ravel()
reg_global = LinearRegression().fit(T_flat.reshape(-1,1), DO_flat)
a_global, b_global = reg_global.coef_[0], reg_global.intercept_
print(f"Global linear fit: DO = {a_global:.4f} × T + {b_global:.4f}")

# Apply to test using T_f_B
OC_f_0_B = a_global * T_f_B + b_global   # (N_te, H)
rmse_lin_B = float(np.sqrt(np.mean((OC_f_0_B - y_do_true)**2)))
print(f"Setup B linear baseline RMSE: {rmse_lin_B:.4f} mg/L")

# Residuals
resid_A = y_do_true - OC_f_0_A   # (N_te, H)
resid_B = y_do_true - OC_f_0_B
print(f"\nResidual stats (Setup A): mean={resid_A.mean():.4f}, std={resid_A.std():.4f}")
print(f"Residual stats (Setup B): mean={resid_B.mean():.4f}, std={resid_B.std():.4f}")

## 4. Setup A — Optuna-Tuned Residual LSTM

In [ ]:
# Features for residual LSTM: [temp, pH, EC, DO, T_f_predicted]
# We need to augment the feature windows with T_f as an additional input

# Build residual targets (training)
_, y_tr_do, _ = make_windows(train_df, train_means, features=FEATURES, targets=['O2C_sensor'])
# Train linear baseline on train set (in-sample T predictions not available — use true T as proxy)
T_tr_proxy = X_tr[:, -1, TEMP_IDX]
OC_f_0_tr = np.column_stack([A_coefs[h,0]*T_tr_proxy + A_coefs[h,1] for h in range(HORIZON)])
resid_tr_A = y_tr_do[:, :, 0] - OC_f_0_tr   # (N_tr, H) residual targets

# Val residual
X_va_do = X_va; y_va_do_true = y_va[:, :, DO_IDX]
T_va_proxy = X_va[:, -1, TEMP_IDX]
OC_f_0_va = np.column_stack([A_coefs[h,0]*T_va_proxy + A_coefs[h,1] for h in range(HORIZON)])
resid_va_A = y_va_do_true - OC_f_0_va

def build_residual_datasets(hidden, n_layers, dropout, batch_size, _seed=SEED):
    """Build scaled RiverDatasets for residual prediction."""
    torch.manual_seed(_seed); np.random.seed(_seed)
    N_tr2, L2, F2 = X_tr.shape; H2 = HORIZON
    # Residual targets shape: (N, H, 1)
    y_res_tr = resid_tr_A[:, :, None].astype('float32')
    y_res_va = resid_va_A[:, :, None].astype('float32')
    y_res_te = (y_do_true - OC_f_0_A)[:, :, None].astype('float32')
    
    fsc = StandardScaler().fit(X_tr.reshape(-1, F2))
    tsc = StandardScaler().fit(y_res_tr.reshape(-1, 1))
    
    def sc(X, y, N):
        Xs = fsc.transform(X.reshape(-1,F2)).reshape(N,L2,F2).astype('float32')
        ys = tsc.transform(y.reshape(-1,1)).reshape(N,H2,1).astype('float32')
        return Xs, ys
    
    Xs_tr2, ys_tr2 = sc(X_tr, y_res_tr, N_tr2)
    Xs_va2, ys_va2 = sc(X_va, y_res_va, X_va.shape[0])
    Xs_te2, ys_te2 = sc(X_te, y_res_te, X_te.shape[0])
    
    return (RiverDataset(Xs_tr2, ys_tr2), 
            RiverDataset(Xs_va2, ys_va2),
            RiverDataset(Xs_te2, ys_te2), tsc)

def objective_A(trial):
    hidden = trial.suggest_categorical('hidden', [64, 128, 256])
    n_layers = trial.suggest_categorical('n_layers', [1, 2])
    dropout = trial.suggest_float('dropout', 0.0, 0.4)
    lr = trial.suggest_float('lr', 1e-4, 5e-3, log=True)
    batch_size = trial.suggest_categorical('batch_size', [64, 128, 256])
    
    ds_tr2, ds_va2, ds_te2, tsc2 = build_residual_datasets(hidden, n_layers, dropout, batch_size)
    dl_tr2 = torch.utils.data.DataLoader(ds_tr2, batch_size=batch_size, shuffle=True, drop_last=True)
    dl_va2 = torch.utils.data.DataLoader(ds_va2, batch_size=batch_size, shuffle=False)
    
    mdl = Seq2SeqLSTM(n_feat=X_tr.shape[2], n_tgt=1, hidden=hidden, n_layers=n_layers, dropout=dropout).to(DEVICE)
    mdl, _ = train_model(mdl, dl_tr2, dl_va2, device=DEVICE, epochs=20, lr=lr, patience=5, verbose=False)
    
    # Predict residual, add back to linear baseline, compute final DO RMSE
    resid_pred = predict(mdl, ds_te2, tsc2, device=DEVICE)[:,:,0]
    oc_final = OC_f_0_A + resid_pred
    rmse_final = float(np.sqrt(np.mean((oc_final - y_do_true)**2)))
    return rmse_final

print(f"Running Optuna Setup A ({N_TRIALS_A} trials)...")
t0 = time.time()
study_A = optuna.create_study(direction='minimize')
study_A.optimize(objective_A, n_trials=N_TRIALS_A, show_progress_bar=False)
print(f"Setup A Optuna done: best RMSE={study_A.best_value:.4f} in {(time.time()-t0)/60:.1f}min")
print(f"Best params: {study_A.best_params}")

## 5. Setup B — Minimal Residual LSTM

In [ ]:
# Setup B: residual targets using global linear baseline
resid_tr_B = y_tr_do[:, :, 0] - (a_global * X_tr[:,-1,TEMP_IDX].reshape(-1,1) + b_global)
resid_va_B = y_va_do_true - (a_global * X_va[:,-1,TEMP_IDX].reshape(-1,1) + b_global)

# Single feature: temperature only
FEAT_B = ['temp_sensor']
feat_b_idx = [list(FEATURES).index('temp_sensor')]
X_tr_b = X_tr[:, :, feat_b_idx]   # (N, L, 1)
X_va_b = X_va[:, :, feat_b_idx]
X_te_b = X_te[:, :, feat_b_idx]

N_b, L_b, F_b = X_tr_b.shape
y_res_tr_b = resid_tr_B[:, :, None].astype('float32')
y_res_va_b = resid_va_B[:, :, None].astype('float32')
y_res_te_b = (y_do_true - OC_f_0_B)[:, :, None].astype('float32')

fsc_b = StandardScaler().fit(X_tr_b.reshape(-1, F_b))
tsc_b = StandardScaler().fit(y_res_tr_b.reshape(-1, 1))

def sc_b(X, y, N):
    Xs = fsc_b.transform(X.reshape(-1,F_b)).reshape(N,L_b,F_b).astype('float32')
    ys = tsc_b.transform(y.reshape(-1,1)).reshape(N,HORIZON,1).astype('float32')
    return Xs, ys

Xs_tr_b, ys_tr_b = sc_b(X_tr_b, y_res_tr_b, N_b)
Xs_va_b, ys_va_b = sc_b(X_va_b, y_res_va_b, X_va_b.shape[0])
Xs_te_b, ys_te_b = sc_b(X_te_b, y_res_te_b, X_te_b.shape[0])

ds_tr_b = RiverDataset(Xs_tr_b, ys_tr_b)
ds_va_b = RiverDataset(Xs_va_b, ys_va_b)
ds_te_b = RiverDataset(Xs_te_b, ys_te_b)

dl_tr_b = torch.utils.data.DataLoader(ds_tr_b, batch_size=BATCH_B, shuffle=True)
dl_va_b = torch.utils.data.DataLoader(ds_va_b, batch_size=BATCH_B, shuffle=False)

mdl_b = Seq2SeqLSTM(n_feat=F_b, n_tgt=1, hidden=HIDDEN_B, n_layers=LAYERS_B, dropout=DROPOUT_B).to(DEVICE)
mdl_b, _ = train_model(mdl_b, dl_tr_b, dl_va_b, device=DEVICE, epochs=30, lr=LR_B, patience=8, verbose=False)

resid_pred_b = predict(mdl_b, ds_te_b, tsc_b, device=DEVICE)[:,:,0]
oc_final_b = OC_f_0_B + resid_pred_b
rmse_b_final = float(np.sqrt(np.mean((oc_final_b - y_do_true)**2)))
print(f"Setup B final DO RMSE: {rmse_b_final:.4f} mg/L")

## 6. Final Results and Comparison

In [ ]:
# Retrain Setup A best model
best = study_A.best_params
ds_tr_A, ds_va_A, ds_te_A, tsc_A = build_residual_datasets(
    best['hidden'], best['n_layers'], best['dropout'], best['batch_size'])
dl_tr_A = torch.utils.data.DataLoader(ds_tr_A, batch_size=best['batch_size'], shuffle=True, drop_last=True)
dl_va_A = torch.utils.data.DataLoader(ds_va_A, batch_size=best['batch_size'], shuffle=False)

mdl_A = Seq2SeqLSTM(n_feat=X_tr.shape[2], n_tgt=1,
                     hidden=best['hidden'], n_layers=best['n_layers'],
                     dropout=best['dropout']).to(DEVICE)
mdl_A, _ = train_model(mdl_A, dl_tr_A, dl_va_A, device=DEVICE, 
                         epochs=30, lr=best['lr'], patience=5, verbose=False)
resid_pred_A = predict(mdl_A, ds_te_A, tsc_A, device=DEVICE)[:,:,0]
oc_final_A = OC_f_0_A + resid_pred_A
rmse_A_final = float(np.sqrt(np.mean((oc_final_A - y_do_true)**2)))

# Summary table
results = pd.DataFrame([
    {'Setup': 'Linear baseline only (A)',  'RMSE': round(rmse_lin_A, 4)},
    {'Setup': 'Linear baseline only (B)',  'RMSE': round(rmse_lin_B, 4)},
    {'Setup': 'Setup A: LSTM T_f + Optuna residual', 'RMSE': round(rmse_A_final, 4)},
    {'Setup': 'Setup B: AR T_f + minimal residual',  'RMSE': round(rmse_b_final, 4)},
    {'Setup': 'Standard LSTM (nb03 best)',            'RMSE': 0.3006},
    {'Setup': 'Ridge regression (nb02)',               'RMSE': 0.3030},
])
print("=" * 55)
print("CASCADED MODEL COMPARISON — Gauge 2473 Test Set")
print("=" * 55)
print(results.to_string(index=False))

results.to_csv(RESULTS_DIR / 'cascaded_do_results.csv', index=False)
print(f"\nSaved: cascaded_do_results.csv")

# Plot
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#D4D1CA','#BAB9B4','#01696F','#0C4E54','#7A7974','#D4D1CA']
bars = ax.barh(results['Setup'], results['RMSE'], color=colors)
ax.axvline(0.3006, color='#01696F', ls='--', lw=1.5, alpha=0.7)
ax.set_xlabel('DO RMSE (mg/L)')
ax.set_title('Cascaded Physics-Informed Model vs Baselines\nGauge 2473, Test Set 2017-2020')
for bar, val in zip(bars, results['RMSE']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'nb18_cascaded_comparison.png', dpi=150)
plt.close()
print("Figure saved: nb18_cascaded_comparison.png")